# Demo interactiva: Q-Learning en rcssserver

Esta es la politica Q-Learning entrenada con la fisica del servidor, semilla 4, evaluada en vivo en el informe (73.5 por ciento en 200 episodios con inicios uniformes). La tabla tiene 41 estados y 4 acciones. Usa el cliente Python RoboCup2DClient del starter para observar y actuar; el entrenador solo coloca al jugador y al balon antes del episodio.

El escenario lateral empieza con jugador en (-15, 0) y balon en (8, 9), a unos 24.7 m. Comparar con el escenario central usando el selector. La ejecucion real se registra y luego se reproduce paso a paso; el porcentaje del informe corresponde a otros inicios y no a una sola corrida.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
OUT = "/workspace/results" if os.path.isdir("/workspace/results") else "../results"


In [ ]:
import asyncio
import io
import socket
import time
from html import escape
import ipywidgets as widgets
from IPython.display import display
from robocup_client import RoboCup2DClient

boton_iniciar = widgets.Button(description="Iniciar demo", button_style="success", icon="play")
boton_detener = widgets.Button(description="Detener ejecucion", button_style="danger", icon="stop", disabled=True)
boton_anterior = widgets.Button(description="Anterior", icon="step-backward", disabled=True)
boton_siguiente = widgets.Button(description="Siguiente", icon="step-forward", disabled=True)
boton_reproducir = widgets.Button(description="Pausar", icon="pause", disabled=True)
escenario = widgets.Dropdown(options=[("Lateral dificil: (8, 9)", (8.0, 9.0)), ("Lateral opuesto: (5, -7)", (5.0, -7.0)), ("Centro original: (0, 0)", (0.0, 0.0))], description="Balon")
velocidad = widgets.FloatSlider(value=1.0, min=0.4, max=3.0, step=0.2, description="Segundos/paso")
indice = widgets.IntSlider(value=0, min=0, max=1, description="Paso", disabled=True)
estado = widgets.HTML(value="Politica de la demo: Q-Learning entrenado con fisica de rcssserver (semilla 4, resultado 73.5 % del informe). Iniciar configura el escenario real.")
detalle = widgets.HTML(value="")
grafica = widgets.Image(format="png", layout=widgets.Layout(width="100%", display="none"))
control = {"detener": False, "activo": False, "pausa": False, "pasos": [], "cliente": None}
display(widgets.VBox([widgets.HBox([boton_iniciar, boton_detener, escenario]), estado, widgets.HBox([boton_anterior, boton_reproducir, boton_siguiente, velocidad]), indice, detalle, grafica]))

def mostrar_paso(cambio=None):
    pasos = control["pasos"]
    if not pasos:
        return
    i = min(indice.value, len(pasos) - 1)
    dato = pasos[i]
    distancias = [p["dist"] if p["dist"] is not None else np.nan for p in pasos]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
    ax1.plot(distancias, color="#c8d4e1", lw=1.5)
    ax1.plot(range(i + 1), distancias[:i + 1], color="#2765a8", lw=2, marker="o", ms=3)
    ax1.axhline(0.8, color="crimson", ls="--", label="Captura: 0.8 m")
    ax1.set(xlabel="Observacion", ylabel="Distancia (m)", title="Distancia recibida del servidor")
    ax1.set_xlim(-0.5, max(2, len(pasos) - 0.5))
    ax1.legend()
    ax1.grid(alpha=0.3)
    d = dato["dist"]
    ax2.scatter([0], [0], s=150, c="#2765a8", label="Jugador")
    if d is not None:
        rad = np.deg2rad(dato["ang"])
        ax2.scatter([d * np.cos(rad)], [d * np.sin(rad)], s=120, c="orange", label="Balon")
        ax2.plot([0, d * np.cos(rad)], [0, d * np.sin(rad)], ls=":", color="gray")
    limite = max(3, np.nanmax(distancias) + 2)
    ax2.set(xlim=(-limite, limite), ylim=(-limite, limite), xlabel="Distancia relativa (m)", ylabel="Distancia relativa (m)", title="Vista relativa del jugador")
    ax2.set_aspect("equal")
    ax2.legend(loc="upper left")
    ax2.grid(alpha=0.3)
    fig.tight_layout()
    imagen = io.BytesIO()
    fig.savefig(imagen, format="png", dpi=110)
    grafica.value = imagen.getvalue()
    grafica.layout.display = "block"
    plt.close(fig)
    columnas = ["Paso", "Ciclo", "Distancia", "Rumbo", "Velocidad", "Resistencia", "Accion"]
    valores = [dato["paso"], dato["tiempo"], f"{d:.2f} m" if d is not None else "No visible", f"{dato['ang']:+.1f}°" if dato["ang"] is not None else "—", f"{dato['speed']:.2f}", f"{dato['stamina']:.0f}", dato["accion"]]
    celdas = "".join(f"<th style='padding:8px'>{escape(str(x))}</th>" for x in columnas)
    datos = "".join(f"<td style='padding:8px;text-align:center'>{escape(str(x))}</td>" for x in valores)
    detalle.value = f"<div>Observacion {i + 1} de {len(pasos)}</div><table border='1' style='border-collapse:collapse'><tr>{celdas}</tr><tr>{datos}</tr></table>"

indice.observe(mostrar_paso, names="value")

async def reproducir(reiniciar=True):
    control["pausa"] = False
    boton_reproducir.description = "Pausar"
    boton_reproducir.icon = "pause"
    if reiniciar:
        indice.value = 0
    mostrar_paso()
    while indice.value < len(control["pasos"]) - 1:
        if control["pausa"]:
            return
        await asyncio.sleep(velocidad.value)
        if not control["pausa"]:
            indice.value += 1
    boton_reproducir.description = "Reproducir"
    boton_reproducir.icon = "play"
    control["pausa"] = True

def enviar_entrenador(sock, addr, comando, respuesta):
    sock.sendto((comando + "\0").encode("ascii"), addr)
    fin = time.monotonic() + 3
    while time.monotonic() < fin:
        try:
            mensaje, _ = sock.recvfrom(16384)
        except socket.timeout:
            continue
        if mensaje.startswith(f"(ok {respuesta}".encode("ascii")):
            return
        if mensaje.startswith(b"(error"):
            raise RuntimeError(mensaje.decode("latin1").strip("\0"))
    raise RuntimeError(f"El entrenador no confirmo {comando}.")

async def ejecutar_demo():
    client = control["cliente"]
    trainer = None
    addr = None
    try:
        q = np.load(os.path.join(OUT, "qlearning_server_seed4.npy"))
        if client is None:
            client = RoboCup2DClient(host=os.getenv("SERVER_HOST", "rcssserver"), port=int(os.getenv("SERVER_PORT", "6000")), team_name="UTEC_QL_Opt")
            estado.value = "Conectando con rcssserver..."
            if not client.connect(init_pos=(-15.0, 0.0)):
                raise RuntimeError("No se pudo conectar con rcssserver.")
            control["cliente"] = client
        trainer = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        trainer.settimeout(0.5)
        trainer.sendto(b"(init (version 15))\0", (client.host, 6001))
        respuesta, addr = trainer.recvfrom(4096)
        if not respuesta.startswith(b"(init ok"):
            raise RuntimeError("Entrenador no disponible: " + respuesta.decode("latin1").strip("\0"))
        x, y = escenario.value
        estado.value = f"Preparando escenario: jugador (-15, 0), balon ({x:g}, {y:g})..."
        enviar_entrenador(trainer, addr, "(change_mode before_kick_off)", "change_mode")
        enviar_entrenador(trainer, addr, f"(move (player UTEC_QL_Opt {client.uniform_number}) -15.000 0.000 0.000 0.000 0.000)", "move")
        enviar_entrenador(trainer, addr, f"(move (ball) {x:.3f} {y:.3f} 0 0.000 0.000)", "move")
        enviar_entrenador(trainer, addr, "(change_mode play_on)", "change_mode")
        estado.value = f"Escenario real: jugador (-15, 0), balon ({x:g}, {y:g}). Esperando vision..."
        await asyncio.sleep(0.3)
        acciones = ["DASH 100", "DASH 50", "GIRAR IZQ", "GIRAR DER"]
        for paso in range(100):
            if control["detener"]:
                break
            obs = client.get_latest_observation()
            dist, ang = obs["ball"] if obs["ball"] is not None else (None, None)
            if dist is None:
                estado_q = 40
            else:
                d_bin = int(np.searchsorted([3.0, 10.0, 20.0], dist, side="right"))
                a_bin = 0 if abs(ang) <= 17.5 else (1 if ang > 0 else 2) if abs(ang) <= 90 else (3 if ang > 0 else 4)
                v_bin = int(obs["speed"] >= 0.2)
                estado_q = (d_bin * 5 + a_bin) * 2 + v_bin
            accion = "CAPTURA" if dist is not None and dist < 0.8 else acciones[int(np.argmax(q[estado_q]))]
            control["pasos"].append({"paso": paso, "tiempo": obs["time"], "dist": dist, "ang": ang, "speed": obs["speed"], "stamina": obs["stamina"], "accion": accion})
            distancia_texto = "no visible" if dist is None else f"{dist:.2f} m"
            estado.value = f"Q-Learning en rcssserver: paso {paso}, balon {distancia_texto}, accion {accion}."
            if accion == "CAPTURA":
                estado.value = f"Captura con Q-Learning: {dist:.2f} m en el paso {paso}. Reproduciendo cada observacion."
                break
            if accion == "DASH 100":
                client.dash(power=100.0)
            elif accion == "DASH 50":
                client.dash(power=50.0)
            elif accion == "GIRAR IZQ":
                client.turn(moment=35.0)
            else:
                client.turn(moment=-35.0)
            await asyncio.sleep(0.1)
        else:
            estado.value = "100 pasos completados. Reproduciendo las observaciones recibidas."
        if control["detener"]:
            estado.value = "Ejecucion detenida. Puedes revisar los pasos registrados."
        if control["pasos"]:
            indice.max = max(1, len(control["pasos"]) - 1)
            indice.disabled = False
            boton_anterior.disabled = False
            boton_siguiente.disabled = False
            boton_reproducir.disabled = False
            asyncio.create_task(reproducir())
    except Exception as e:
        estado.value = f"Error: {e}"
    finally:
        if trainer is not None:
            try:
                if addr is not None:
                    trainer.sendto(b"(bye)\0", addr)
            except OSError:
                pass
            trainer.close()
        control["activo"] = False
        boton_iniciar.disabled = False
        escenario.disabled = False
        boton_detener.disabled = True

def iniciar(_):
    if control["activo"]:
        return
    control["activo"] = True
    control["detener"] = False
    control["pasos"] = []
    boton_iniciar.disabled = True
    escenario.disabled = True
    boton_detener.disabled = False
    indice.disabled = True
    boton_anterior.disabled = True
    boton_siguiente.disabled = True
    boton_reproducir.disabled = True
    detalle.value = ""
    grafica.layout.display = "none"
    asyncio.create_task(ejecutar_demo())

def detener(_):
    control["detener"] = True

def anterior(_):
    control["pausa"] = True
    boton_reproducir.description = "Reproducir"
    boton_reproducir.icon = "play"
    indice.value = max(0, indice.value - 1)

def siguiente(_):
    control["pausa"] = True
    boton_reproducir.description = "Reproducir"
    boton_reproducir.icon = "play"
    indice.value = min(len(control["pasos"]) - 1, indice.value + 1)

def alternar_reproduccion(_):
    if control["pausa"]:
        asyncio.create_task(reproducir(reiniciar=indice.value == len(control["pasos"]) - 1))
    else:
        control["pausa"] = True
        boton_reproducir.description = "Reproducir"
        boton_reproducir.icon = "play"

boton_iniciar.on_click(iniciar)
boton_detener.on_click(detener)
boton_anterior.on_click(anterior)
boton_siguiente.on_click(siguiente)
boton_reproducir.on_click(alternar_reproduccion)
